# Brakke's samples: drops and capillarity


Liquid volumes against walls: contact angles, gravity, curved tanks, and the forces a
liquid bridge exerts.

Each sample is run twice from its datafile: once with the recipe Brakke wrote into
it (its `gogo` command, or the usage its comments describe), once with a default
`ev.relax()` given as many refinements as the recipe does (`levels`). The table
shows the energy each ends with, the residual (how far from equilibrium, see
[Relaxing a surface](relaxing.md)), the facet count, the time, and what `relax()`
reports: converged, stable, and any health issues. A lower energy on the same mesh
is not always the better answer: a coarse mesh can lower its energy in ways the
smooth surface can't, so where an exact value is known, the error against it is
the measure. Times are from the documentation build and vary.

In [ ]:
import warnings
import numpy as np
import pyvista as pv
import pysurfaceevolver as pyse
import brakke_helpers as bh

pv.set_jupyter_backend("static")
warnings.simplefilter("ignore", pyse.EvolverWarning)

## `mound.fe`: a drop on a table

A drop of fixed volume on the plane $z = 0$ with a contact angle (here 90 degrees, a
parameter) and gravity. The contact angle comes from an energy integral on the
contact line; the table top is drawn with fixed facets. `re` refines the contact line
only.

In [ ]:
print(bh.recipe("mound.fe"))
runs = bh.compare("mound.fe")
bh.show(runs)

The recipe's `re` refines the edges on the table first, so its mesh has more facets
there; the energies agree to about 1e-5. The file's other recipes change the contact
angle and gravity; `gogo2` sets 45 degrees:

In [ ]:
runs = bh.compare("mound.fe", "gogo2", levels=2,
                  setup=lambda ev: ev.command("angle := 45"))
bh.show(runs)

## `addload_example.fe`: nine drops at once

On loading, the file's `setup` script loads the mound eight more times next to the
first (`addload`), each drop with its own contact angle on its edges (from 50 to 130
degrees).

In [ ]:
runs = bh.compare("addload_example.fe")
bh.show(runs)

## `column.fe`: a column of solder

A liquid solder column between two pads, its upper pad offset sideways (a skewed
catenoid shape with a volume and gravity; units cgs). The file goes on to compute the
forces on the pads by central differences.

In [ ]:
runs = bh.compare("column.fe")
bh.show(runs)

## `sphere.fe`: liquid in a spherical tank

A spherical tank partly filled, with a contact angle on the wall. Straight edges can't
follow the curved wall, so the file gives the gaps between them and the wall an
energy and volume (a "convex" constraint). Evolver has no Hessian for that gap energy,
so `relax()` uses gradient steps only; the recipe uses conjugate gradients at the
end.

In [ ]:
print(bh.recipe("sphere.fe"))
runs = bh.compare("sphere.fe")
bh.show(runs)

Neither run reaches a residual near zero: gradient steps converge slowly, and the
energies agree to about 3e-4.

## `tankex.fe`: liquid in a cylindrical tank

A flat-ended cylindrical tank on its side, gravity pulling the liquid to the wall, gap
energies as in `sphere.fe`. The file has no recipe; here two refinements and plain
gradient steps (it starts with 8 facets, too coarse to keep its shape on its own).
With no Hessian, `relax()` goes on with gradient rounds while no facet is degenerate;
from fewer refinements (`levels=1`) its facets degenerate first, and it stops there and
says so instead of sliding into what the degenerate mesh allows.

In [ ]:
runs = bh.compare("tankex.fe", "r; r; g 1000", levels=2)
bh.show(runs)